# Discrete-event systems with StormPy

In [ ]:
import json
import random
import pandas as pd
import stormpy
import stormpy.simulator as ss

## DTMC: User Engagement

In [ ]:
def load_model(filename, verbose=True,  prism_compat=False):
    """Load a PRISM model from a file and build a StormPy model."""
    program = stormpy.parse_prism_program(str(filename), prism_compat=prism_compat)
    options = stormpy.BuilderOptions()
    options.set_build_state_valuations(True)
    options.set_build_choice_labels(True)
    options.set_build_all_labels()
    options.set_build_all_reward_models()
    model = stormpy.build_sparse_model_with_options(program, options)
    if verbose:
        print(model)
    return model


engagement_dtmc = load_model("./engagement.pm")
# --------------------------------------------------------------
# Model type: 	DTMC (sparse)
# States: 	5
# Transitions: 	12
# Reward Models:  none
# State Labels: 	9 labels
#    * deadlock -> 0 item(s)
#    * disengaged -> 1 item(s)
#    * success -> 1 item(s)
#    * converted -> 1 item(s)
#    * init -> 1 item(s)
#    * failure -> 1 item(s)
#    * engaged -> 1 item(s)
#    * browsing -> 1 item(s)
#    * abandoned -> 1 item(s)
# Choice Labels: 	0 labels
# --------------------------------------------------------------

In [ ]:
def simulate_path(model, *, max_steps=10, seed=42, stop_labels=()):
    """Simulate a path through the model, returning a DataFrame with the results."""
    rng = random.Random(seed)
    simulator = ss.create_simulator(model, seed=seed)
    state_id, _, labels = simulator.restart()
    stop_labels = set(stop_labels)
    action_label = "—"
    rows = []

    for step in range(max_steps + 1):
        rows.append(
            {
                "step": step,
                "action": action_label,
                **json.loads(str(model.state_valuations.get_json(state_id))),
                "labels": ", ".join(sorted(labels - {"init"})),
            }
        )
        if stop_labels & labels or simulator.is_done():
            break

        action = rng.randrange(simulator.nr_available_actions())
        choice = model.transition_matrix.get_row_group_start(state_id) + action
        action_label = (
            ", ".join(sorted(model.choice_labeling.get_labels_of_choice(choice))) or "—"
        )
        state_id, _, labels = simulator.step(action)

    return pd.DataFrame(rows)

In [ ]:
df = simulate_path(
    engagement_dtmc, max_steps=10, seed=7, stop_labels={"converted", "abandoned"}
)

print(df)
#    step action  state              labels
# 0     0      —      0            browsing
# 1     1      —      1             engaged
# 2     2      —      1             engaged
# 3     3      —      3  converted, success

## MDP: Activity Recommendation Agent

In [ ]:
activity_mdp = load_model("activity_agent.pm")
# -------------------------------------------------------------- 
# Model type: 	MDP (sparse)
# States: 	13
# Transitions: 	27
# Choices: 	15
# Reward Models:  task_completion
# State Labels: 	9 labels
#    * deadlock -> 0 item(s)
#    * s_success -> 1 item(s)
#    * s_abandon -> 4 item(s)
#    * s_W -> 1 item(s)
#    * s_WM -> 1 item(s)
#    * s_M -> 1 item(s)
#    * init -> 1 item(s)
#    * s_0 -> 1 item(s)
#    * done -> 4 item(s)
# Choice Labels: 	7 labels
#    * ask_weather -> 2 item(s)
#    * ask_both -> 1 item(s)
#    * complete -> 1 item(s)
#    * ask_mood -> 2 item(s)
#    * terminate -> 4 item(s)
#    * recommend -> 1 item(s)
#    * done -> 4 item(s)
# --------------------------------------------------------------

In [ ]:
df = simulate_path(activity_mdp, max_steps=10, seed=0, stop_labels={"done"})

print(df)
#    step       action  mood  status  weather     labels
# 0     0            —     0       0        0        s_0
# 1     1     ask_mood     1       0        0        s_M
# 2     2  ask_weather     1       0        1       s_WM
# 3     3    recommend     1       2        1  s_abandon
# 4     4    terminate     1       3        1       done

## CTMC: timed user engagement

In [ ]:
engagement_ctmc = load_model("engagement_timed.pm", prism_compat=True)
# -------------------------------------------------------------- 
# Model type: 	CTMC (sparse)
# States: 	5
# Transitions: 	10
# Reward Models:  none
# State Labels: 	8 labels
#    * init -> 1 item(s)
#    * engaged -> 1 item(s)
#    * deadlock -> 2 item(s)
#    * disengaged -> 1 item(s)
#    * converted -> 1 item(s)
#    * done -> 2 item(s)
#    * browsing -> 1 item(s)
#    * abandoned -> 1 item(s)
# Choice Labels: 	0 labels
# --------------------------------------------------------------

In [ ]:
def simulate_ctmc_path(model, *, max_steps=10, seed=42, stop_labels=()):
    rng = random.Random(seed)
    simulator = ss.create_simulator(model, seed=seed)
    state_id, _, labels = simulator.restart()
    stop_labels = set(stop_labels)
    elapsed_time = 0.0
    rows = []

    for step in range(max_steps + 1):
        rows.append(
            {
                "step": step,
                "time": elapsed_time,
                "action": "—",
                **json.loads(str(model.state_valuations.get_json(state_id))),
                "labels": ", ".join(sorted(labels - {"init"})),
            }
        )
        if stop_labels & labels or simulator.is_done():
            break

        action = rng.randrange(simulator.nr_available_actions())
        elapsed_time += rng.expovariate(float(model.exit_rates[state_id]))
        state_id, _, labels = simulator.step(action)

    return pd.DataFrame(rows)

In [ ]:
df = simulate_ctmc_path(
    engagement_ctmc, max_steps=20, seed=19, stop_labels={"converted", "abandoned"}
)

print(df)
#    step      time action  state                     labels
# 0     0  0.000000      —      0                   browsing
# 1     1  2.364162      —      1                    engaged
# 2     2  3.956159      —      3  converted, deadlock, done